In [1]:
# ==========================================
# Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [2]:
# ==========================================
# Load Datasets
# ==========================================

application = pd.read_csv('/content/application_record.csv')

credit = pd.read_csv('/content/credit_record.csv')

print("Datasets Loaded Successfully!")

Datasets Loaded Successfully!


In [3]:
print("Application Dataset :", application.shape)

print("Credit Dataset :", credit.shape)

Application Dataset : (270805, 18)
Credit Dataset : (1048575, 3)


In [4]:
print("Application Duplicate Records :", application.duplicated().sum())

print("Credit Duplicate Records :", credit.duplicated().sum())

Application Duplicate Records : 0
Credit Duplicate Records : 0


In [5]:
application = application.drop_duplicates()

credit = credit.drop_duplicates()

print("Duplicates Removed Successfully!")

Duplicates Removed Successfully!


In [6]:
print(application.duplicated().sum())

print(credit.duplicated().sum())

0
0


Detect and Handle Missing Values

In [7]:
# ==========================================
# Check Missing Values
# ==========================================

application.isnull().sum()

,0
ID,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0


In [8]:
missing_percentage = (
    application.isnull().sum() /
    len(application)
) * 100

missing_percentage.sort_values(ascending=False)

,0
OCCUPATION_TYPE,30.629420
FLAG_EMAIL,0.000369
CNT_FAM_MEMBERS,0.000369
FLAG_PHONE,0.000369
FLAG_WORK_PHONE,0.000369
ID,0.000000
CODE_GENDER,0.000000
FLAG_OWN_CAR,0.000000
CNT_CHILDREN,0.000000
FLAG_OWN_REALTY,0.000000


In [9]:
missing_df = pd.DataFrame({
    'Missing Values': application.isnull().sum(),
    'Percentage': missing_percentage
})

missing_df[missing_df['Missing Values'] > 0]

,Missing Values,Percentage
FLAG_WORK_PHONE,1,0.000369
FLAG_PHONE,1,0.000369
FLAG_EMAIL,1,0.000369
OCCUPATION_TYPE,82946,30.629420
CNT_FAM_MEMBERS,1,0.000369


In [10]:
application['DAYS_EMPLOYED'].replace(
    365243,
    np.nan,
    inplace=True
)

In [11]:
(application['DAYS_EMPLOYED'] == 365243).sum()

np.int64(0)

In [12]:
application['OCCUPATION_TYPE'].fillna(
    'Unknown',
    inplace=True
)

In [13]:
application.isnull().sum()

,0
ID,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0


Data Cleaning and Merging

In [14]:
# Check unique status values
credit['STATUS'].value_counts()

,count
STATUS,
C,442031
0,383120
X,209230
1,11090
5,1693
2,868
3,320
4,223


In [15]:
# Create binary target variable
credit['TARGET'] = credit['STATUS'].apply(
    lambda x: 1 if x in ['2', '3', '4', '5'] else 0
)

credit[['STATUS', 'TARGET']].head()

,STATUS,TARGET
0,X,0
1,0,0
2,0,0
3,0,0
4,C,0


In [16]:
credit_target = credit.groupby('ID')['TARGET'].max().reset_index()

credit_target.head()

,ID,TARGET
0,5001711,0
1,5001712,0
2,5001713,0
3,5001714,0
4,5001715,0


In [17]:
data = application.merge(
    credit_target,
    on='ID',
    how='inner'
)

print("Merged Dataset Shape:", data.shape)

Merged Dataset Shape: (36105, 19)


In [18]:
data.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542.0,1,1.0,0.0,0.0,Unknown,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542.0,1,1.0,0.0,0.0,Unknown,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134.0,1,0.0,0.0,0.0,Security staff,2.0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051.0,1,0.0,1.0,1.0,Sales staff,1.0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051.0,1,0.0,1.0,1.0,Sales staff,1.0,0


In [19]:
from sklearn.preprocessing import LabelEncoder
import joblib

categorical_columns = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

encoders = {}

for column in categorical_columns:
    le = LabelEncoder()
    le.fit(data[column].astype(str))
    encoders[column] = le

joblib.dump(encoders, "encoders.pkl")

print("Encoders created successfully!")
print(encoders["CODE_GENDER"].classes_)

Encoders created successfully!
['F' 'M']


In [20]:
data['TARGET'].value_counts()

,count
TARGET,
0,35841
1,264


In [21]:
data.isnull().sum()

,0
ID,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0


Feature Engineering

In [22]:
data.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542.0,1,1.0,0.0,0.0,Unknown,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542.0,1,1.0,0.0,0.0,Unknown,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134.0,1,0.0,0.0,0.0,Security staff,2.0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051.0,1,0.0,1.0,1.0,Sales staff,1.0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051.0,1,0.0,1.0,1.0,Sales staff,1.0,0


In [23]:
# Convert age from days to years
data['AGE'] = (-data['DAYS_BIRTH'] / 365).astype(int)

data[['DAYS_BIRTH', 'AGE']].head()

,DAYS_BIRTH,AGE
0,-12005,32
1,-12005,32
2,-21474,58
3,-19110,52
4,-19110,52


In [24]:
# Convert employment days into years
data['EMPLOYMENT_YEARS'] = (-data['DAYS_EMPLOYED'] / 365)

data[['DAYS_EMPLOYED', 'EMPLOYMENT_YEARS']].head()

,DAYS_EMPLOYED,EMPLOYMENT_YEARS
0,-4542.0,12.443836
1,-4542.0,12.443836
2,-1134.0,3.106849
3,-3051.0,8.358904
4,-3051.0,8.358904


In [25]:
data['EMPLOYMENT_YEARS'].fillna(
    data['EMPLOYMENT_YEARS'].median(),
    inplace=True
)

In [26]:
data.drop(
    columns=[
        'ID',
        'DAYS_BIRTH',
        'DAYS_EMPLOYED'
    ],
    inplace=True
)

In [27]:
data.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET,AGE,EMPLOYMENT_YEARS
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1.0,0.0,0.0,Unknown,2.0,0,32,12.443836
1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1.0,0.0,0.0,Unknown,2.0,0,32,12.443836
2,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0.0,0.0,0.0,Security staff,2.0,0,58,3.106849
3,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0.0,1.0,1.0,Sales staff,1.0,0,52,8.358904
4,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0.0,1.0,1.0,Sales staff,1.0,0,52,8.358904


In [28]:
data.isnull().sum()

,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0
FLAG_MOBIL,0


Encode Features and Prepare Data for Machine Learning

In [29]:
# Check data types
data.dtypes

,0
CODE_GENDER,object
FLAG_OWN_CAR,object
FLAG_OWN_REALTY,object
CNT_CHILDREN,int64
AMT_INCOME_TOTAL,float64
NAME_INCOME_TYPE,object
NAME_EDUCATION_TYPE,object
NAME_FAMILY_STATUS,object
NAME_HOUSING_TYPE,object
FLAG_MOBIL,int64


In [30]:
# Select categorical columns
categorical_columns = data.select_dtypes(include='object').columns

print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
Index(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE',
       'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
       'OCCUPATION_TYPE'],
      dtype='object')


In [31]:
from sklearn.preprocessing import LabelEncoder
import joblib

# Dictionary to store encoders
encoders = {}

# Encode each categorical column
for column in categorical_columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])
    encoders[column] = le

print("Categorical Features Encoded Successfully!")

Categorical Features Encoded Successfully!


In [32]:
# Save all encoders
joblib.dump(encoders, "encoders.pkl")

print("Encoders saved successfully!")

Encoders saved successfully!


In [33]:
data.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET,AGE,EMPLOYMENT_YEARS
0,1,1,1,0,427500.0,4,1,0,4,1,1.0,0.0,0.0,17,2.0,0,32,12.443836
1,1,1,1,0,427500.0,4,1,0,4,1,1.0,0.0,0.0,17,2.0,0,32,12.443836
2,1,1,1,0,112500.0,4,4,1,1,1,0.0,0.0,0.0,16,2.0,0,58,3.106849
3,0,0,1,0,270000.0,0,4,3,1,1,0.0,1.0,1.0,14,1.0,0,52,8.358904
4,0,0,1,0,270000.0,0,4,3,1,1,0.0,1.0,1.0,14,1.0,0,52,8.358904


In [34]:
X = data.drop('TARGET', axis=1)
y = data['TARGET']

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (36105, 17)
Target Shape : (36105,)


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)

Training Features : (28884, 17)
Testing Features : (7221, 17)


In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature Scaling Completed Successfully!")

Feature Scaling Completed Successfully!


In [37]:
import joblib

joblib.dump(scaler, "scaler.pkl")

print("Scaler Saved Successfully!")

Scaler Saved Successfully!


In [38]:
data.to_csv("processed_credit_card_data.csv", index=False)

print("Processed Dataset Saved Successfully!")

Processed Dataset Saved Successfully!


In [39]:
data = pd.read_csv('/content/processed_credit_card_data.csv')

print(data["TARGET"].value_counts())

TARGET
0    35841
1      264
Name: count, dtype: int64


In [40]:
print(data["CODE_GENDER"].unique())
print(data["FLAG_OWN_CAR"].unique())
print(data["NAME_INCOME_TYPE"].unique())

[1 0]
[1 0]
[4 0 1 2 3]
